# Maps in Python Plotly

This notebook demonstrates how to use Plotly to create interactive maps and describe the current state with geospatial data.

## 1. Import Required Libraries

In [21]:
import pandas as pd
import plotly.express as px

## 2. Load a Geospatial Dataset

We'll use a sample dataset with country locations and values. Replace with your own data as needed.

In [22]:
import requests

# Fetch the data.
df = pd.read_csv("https://ourworldindata.org/grapher/carbon-intensity-electricity.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/carbon-intensity-electricity.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()

In [23]:
df

,entity,code,year,co2_intensity__gco2_kwh
0,ASEAN (Ember),NaN,2000,572.52
1,ASEAN (Ember),NaN,2001,570.24
2,ASEAN (Ember),NaN,2002,572.68
3,ASEAN (Ember),NaN,2003,577.02
4,ASEAN (Ember),NaN,2004,583.61
...,...,...,...,...
6101,Zimbabwe,ZWE,2019,429.25
6102,Zimbabwe,ZWE,2020,371.09
6103,Zimbabwe,ZWE,2021,269.55
6104,Zimbabwe,ZWE,2022,297.54


## 3. Exploratory Data Analysis (EDA)

In [25]:
# EDA: Explore the DataFrame
# print("Shape:", df.shape)
# print("Columns:", df.columns.tolist())
# print("Sample rows:")
# print(df.head())

# print("\nMissing values per column:")
# print(df.isnull().sum())

# if 'region' in df.columns:
#     print("\nUnique regions:", df['region'].unique())
# if 'country' in df.columns:
#     print("\nUnique countries:", df['country'].nunique())

# print("\nSummary statistics:")
print(df.describe(include='all'))

         entity  code         year  co2_intensity__gco2_kwh
count      6106  5805  6106.000000              6106.000000
unique      238   226          NaN                      NaN
top     Belgium   LVA          NaN                      NaN
freq         36    36          NaN                      NaN
mean        NaN   NaN  2010.808385               478.178851
std         NaN   NaN     7.933547               241.595818
min         NaN   NaN  1990.000000                 0.000000
25%         NaN   NaN  2005.000000               297.015000
50%         NaN   NaN  2011.000000               525.260000
75%         NaN   NaN  2017.000000               650.270000
max         NaN   NaN  2025.000000              1306.720000


## 3. Create a Choropleth Map

Visualize values by country using color.

In [32]:
# Animated choropleth map: Carbon Intensity by Country with Year Slider
df_anim = df_anim.sort_values(by=year_col)[(df_anim.year >= 2000) & (df_anim.year <= 2022)]


fig = px.choropleth(
    df_anim,
    locations='code',
    color=carbon_col,
    hover_name='entity',
    animation_frame=year_col,
    color_continuous_scale=px.colors.diverging.RdYlGn_r,
    title='Poland doesn`t seem to be impacted by decrease in carbon intensity from neighbour countries',
    labels={carbon_col: 'gCO₂/kWh'}
)

fig.update_layout(
    height=600,
    width=1000,
    coloraxis_colorbar=dict(title='gCO₂/kWh'),
    geo=dict(
        projection_type='natural earth',
        center=dict(lat=54, lon=15),  # Center of Europe
        projection_scale=3.5,         # Zoom level
        showland=True,
        landcolor='lightgray',
        showocean=True,
        oceancolor='lightblue',
        showcoastlines=True,
        coastlinecolor='white',
    )
)

fig.show()

/var/folders/12/bl9k44cn4g928c_73g5pt2cw0000gn/T/ipykernel_66951/2072499097.py:2: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



## Describe Spatial Patterns and the Current State

Write a short summary of what the map shows.

In [ ]:
# Define column names
carbon_col = 'co2_intensity__gco2_kwh'
year_col = 'year'

# Filter data for European countries and prepare year-over-year changes
european_countries = ['FRA', 'DEU', 'GBR', 'ITA', 'ESP', 'POL', 'NLD', 'BEL', 'GRC', 'PRT', 'AUT', 'SVK', 'CZE', 'HUN', 'ROU', 'BGR', 'HRV', 'SVN', 'SRB', 'UKR', 'NOR', 'SWE', 'DNK', 'FIN']

# Filter for Europe and years 2000-2022
df_europe = df[df['code'].isin(european_countries)].copy()

# Pivot to get countries as rows and years as columns
df_pivot = df_europe.pivot_table(index='code', columns=year_col, values=carbon_col)

# Calculate year-over-year changes (percentage change)
yoy_changes = df_pivot.pct_change(axis=1) * 100

# Create heatmap
fig_heatmap = px.imshow(
    yoy_changes,
    labels=dict(x='Year', y='Country Code', color='Change (%)'),
    color_continuous_scale='RdBu_r',
    title='Year-over-Year Percentage Changes in Carbon Intensity - European Countries',
    aspect='auto',
    height=600,
    width=1200
)

fig_heatmap.update_xaxes(side='bottom')
fig_heatmap.show()

In [39]:
# Define column names
carbon_col = 'co2_intensity__gco2_kwh'
year_col = 'year'

# Filter data for European countries and prepare year-over-year changes
european_countries = ['FRA', 'DEU', 'GBR', 'ITA', 'ESP', 'POL', 'NLD', 'BEL', 'GRC', 'PRT', 'AUT', 'SVK', 'CZE', 'HUN', 'ROU', 'BGR', 'HRV', 'SVN', 'SRB', 'UKR', 'NOR', 'SWE', 'DNK', 'FIN']

# Filter for Europe and years 2000-2022
df_europe = df[df['code'].isin(european_countries)].copy()

# Pivot to get countries as rows and years as columns
df_pivot = df_europe.pivot_table(index='code', columns=year_col, values=carbon_col)

# Calculate year-over-year percentage changes
yoy_changes = df_pivot.pct_change(axis=1) * 100

# Calculate average year-over-year change and latest CO2 intensity for each country
avg_yoy = yoy_changes.mean(axis=1)
latest_co2 = df_pivot.iloc[:, -1]

# Create scatter plot
fig_scatter = px.scatter(
    x=latest_co2,
    y=avg_yoy,
    labels={'x': 'Latest CO₂ Intensity (gCO₂/kWh)', 'y': 'Average Year-over-Year Change (%)'},
    title='Average YoY % Change vs Latest CO₂ Intensity - European Countries',
    hover_name=latest_co2.index,
    width=800,
    height=600
)

fig_scatter.update_traces(marker=dict(size=12, opacity=0.7))
fig_scatter.show()

/var/folders/12/bl9k44cn4g928c_73g5pt2cw0000gn/T/ipykernel_66951/123746898.py:15: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [41]:
# Calculate average year-over-year change and latest CO2 intensity for each country
avg_yoy = yoy_changes.mean(axis=1)
latest_co2 = df_pivot.iloc[:, -1]

# Calculate median values for quadrant division
median_co2 = latest_co2.median()
median_yoy = avg_yoy.median()

# Create scatter plot with quadrant lines
fig_scatter = px.scatter(
    x=latest_co2,
    y=avg_yoy,
    labels={'x': 'Latest CO₂ Intensity (gCO₂/kWh)', 'y': 'Average YoY Change (%) — negative = improving'},
    title='Poland and Serbia are lagging behind in decarbonization compared to their European peers',
    hover_name=latest_co2.index,
    width=900,
    height=600
)

fig_scatter.update_traces(marker=dict(size=12, opacity=0.7))

# Add vertical and horizontal lines to divide into quadrants
fig_scatter.add_vline(x=median_co2, line_dash="dash", line_color="gray", opacity=0.5)
fig_scatter.add_hline(y=median_yoy, line_dash="dash", line_color="gray", opacity=0.5)

# Quadrant logic:
# x > median_co2 → High current intensity (bad)
# x < median_co2 → Low current intensity (good)
# y > median_yoy → Less decrease / increasing (bad)
# y < median_yoy → More decrease (good)
x_low = latest_co2.min() + (median_co2 - latest_co2.min()) * 0.5
x_high = median_co2 + (latest_co2.max() - median_co2) * 0.5
y_low = avg_yoy.min() + (median_yoy - avg_yoy.min()) * 0.5
y_high = median_yoy + (avg_yoy.max() - median_yoy) * 0.5

fig_scatter.add_annotation(x=x_low,  y=y_high, text="✅ Low intensity<br>Slow improvement",  showarrow=False, opacity=0.6, bgcolor="lightyellow")
fig_scatter.add_annotation(x=x_low,  y=y_low,  text="🏆 Low intensity<br>Fast improvement",  showarrow=False, opacity=0.6, bgcolor="lightgreen")
fig_scatter.add_annotation(x=x_high, y=y_high, text="⚠️ High intensity<br>Slow improvement", showarrow=False, opacity=0.6, bgcolor="lightsalmon")
fig_scatter.add_annotation(x=x_high, y=y_low,  text="📉 High intensity<br>Fast improvement", showarrow=False, opacity=0.6, bgcolor="lightyellow")

fig_scatter.show()